In [4]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import os

# 导入我们的模块
from config import Config
from data_generator import SingleModeDualWavelengthDataGenerator
from model import SimpleMultiWavelengthModel
from trainer import SimpleTrainer

def main():
    # Create configuration
    config = Config(
        field_size=128,
        layer_size=128,
        batch_size=4,
        wavelengths=[450e-9, 650e-9],  # Only two wavelengths: blue and red
        learning_rate=1e-3,
        num_epochs=100,
        save_dir='./single_mode_dual_wavelength_results'
    )
    
    # Create save directory
    os.makedirs(config.save_dir, exist_ok=True)
    
    # 打印配置信息
    config.print_config()
    
    # 创建数据生成器
    print("Creating data generator...")
    data_generator = SingleModeDualWavelengthDataGenerator(config)
    
    # 可视化分离概念
    print("Generating separation concept visualization...")
    concept_path = os.path.join(config.save_dir, 'separation_concept.png')
    data_generator.visualize_separation_concept(save_path=concept_path)
    
    # 可视化检测器布局
    print("Generating detector layout visualization...")
    layout_path = os.path.join(config.save_dir, 'detector_layout.png')
    data_generator.visualize_detector_layout(save_path=layout_path)
    
    # 创建模型
    print("Creating model...")
    model = SimpleMultiWavelengthModel(config)
    print(f"Model created with {model.num_layers} layers")
    print(f"Total parameters: {sum(p.numel() for p in model.parameters())}")
    
    # 创建训练器
    trainer = SimpleTrainer(config, data_generator)
    
    # Train model
    print("Starting training...")
    train_result = trainer.train_model(model, num_epochs=config.num_epochs)
    
    # 从结果字典中提取信息
    trained_model = train_result['model']
    losses = train_result['losses']
    visibility = train_result['visibility']
    training_time = train_result['training_time']
    
    print(f"Training completed!")
    print(f"Final loss: {losses[-1]:.6f}")
    print(f"Final visibility: {visibility:.4f}")
    print(f"Training time: {training_time:.2f} seconds")
    
    # Save model
    model_path = os.path.join(config.save_dir, 'trained_model.pth')
    torch.save(trained_model.state_dict(), model_path)
    print(f"Model saved to: {model_path}")
    
    # 计算分离性能指标
    print("\nCalculating wavelength separation metrics...")
    separation_metrics = trainer.calculate_wavelength_separation(trained_model)
    
    # 打印分离指标
    print("\nWavelength Separation Performance:")
    print("-" * 50)
    for wl_key, metrics in separation_metrics.items():
        if wl_key != 'overall':
            print(f"{wl_key}:")
            print(f"  Efficiency: {metrics['efficiency']:.4f}")
            print(f"  Avg Crosstalk: {metrics['avg_crosstalk']:.4f}")
            print(f"  SNR: {metrics['snr']:.2f}")
            print(f"  Contrast: {metrics['contrast']:.4f}")
    
    print(f"\nOverall Performance:")
    overall = separation_metrics['overall']
    print(f"  Average Efficiency: {overall['avg_efficiency']:.4f} ± {overall['efficiency_std']:.4f}")
    print(f"  Average Crosstalk: {overall['avg_crosstalk']:.4f} ± {overall['crosstalk_std']:.4f}")
    print(f"  Separation Ratio: {overall['separation_ratio']:.2f}")
    
    # ===== 使用 SimpleVisualizer 进行可视化 =====
    print("\nGenerating result visualizations...")
    
    # 创建 SimpleVisualizer 实例
    from visualizer import SimpleVisualizer
    visualizer = SimpleVisualizer(config)
    
    # 1. 可视化训练损失 - 使用正确的方法名
    loss_path = os.path.join(config.save_dir, 'training_loss.png')
    visualizer.plot_loss_curve(losses, "训练损失曲线", save_path=loss_path)
    
    # 2. 可视化检测器区域
    detector_path = os.path.join(config.save_dir, 'detector_regions.png')
    visualizer.plot_detector_regions(save_path=detector_path)
    
    # 3. 可视化相位掩模
    masks_path = os.path.join(config.save_dir, 'phase_masks.png')
    visualizer.plot_phase_masks(trained_model.phase_masks, save_path=masks_path)
    
    # 4. 生成输入输出场并可视化
    with torch.no_grad():
        input_fields = data_generator.generate_input_fields()
        if isinstance(input_fields, list):
            device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
            input_fields = [field.to(device) for field in input_fields]
        
        output_fields = trained_model(input_fields)
    
    # 5. 可视化能量分布
    energy_path = os.path.join(config.save_dir, 'energy_distributions.png')
    visualizer.plot_energy_distributions(output_fields, config.wavelengths, save_path=energy_path)
    
    # 6. 可视化目标区域
    target_path = os.path.join(config.save_dir, 'target_regions.png')
    visualizer.plot_target_regions(output_fields, config.wavelengths, 
                                 config.offsets, config.detect_size, save_path=target_path)
    
    # 7. 可视化光场传播过程
    propagation_path = os.path.join(config.save_dir, 'field_propagation.png')
    visualizer.plot_field_propagation(trained_model, input_fields, save_path=propagation_path)
    
    # 8. 可视化分离性能指标
    separation_path = os.path.join(config.save_dir, 'separation_performance.png')
    visualizer.plot_wavelength_separation_metrics(separation_metrics, save_path=separation_path)
    
    print(f"\nAll results saved to: {config.save_dir}")
    print("Training and visualization completed successfully!")
    
    # 保存结果摘要
    results_summary = {
        'config': {
            'field_size': config.field_size,
            'layer_size': config.layer_size,
            'wavelengths_nm': [int(wl*1e9) for wl in config.wavelengths],
            'offsets': config.offsets,
            'detectsize': config.detectsize,
            'num_layers': config.num_layers,
            'epochs': config.epochs,
            'batch_size': config.batch_size,
            'learning_rate': config.learning_rate
        },
        'training_results': {
            'final_loss': losses[-1],
            'final_visibility': visibility,
            'training_time': training_time,
            'total_epochs': len(losses)
        },
        'separation_metrics': separation_metrics
    }
    
    # 保存结果摘要
    import json
    summary_path = os.path.join(config.save_dir, 'results_summary.json')
    with open(summary_path, 'w') as f:
        # 将numpy数组转换为列表以便JSON序列化
        def convert_numpy(obj):
            if isinstance(obj, np.ndarray):
                return obj.tolist()
            elif isinstance(obj, np.integer):
                return int(obj)
            elif isinstance(obj, np.floating):
                return float(obj)
            return obj
        
        json.dump(results_summary, f, indent=2, default=convert_numpy)
    
    print(f"Results summary saved to: {summary_path}")

if __name__ == "__main__":
    main()


Configuration validation completed.
CONFIGURATION SUMMARY
Field size: 128 x 128 pixels
Layer size: 128 x 128 pixels
Pixel size: 1.0 μm
Wavelengths: [450, 650] nm
Detection offsets: [(-10, 0), (10, 0)]
Detection size: 10 x 10 pixels
Number of layers: 3
Training epochs: 100
Batch size: 4
Learning rate: 0.001
Device: cuda
Save directory: ./single_mode_dual_wavelength_results
Creating data generator...
MMF data file not found, generating Gaussian fundamental mode
Generating separation concept visualization...
Separation concept visualization saved to: ./single_mode_dual_wavelength_results/separation_concept.png
Generating detector layout visualization...
Detector layout visualization saved to: ./single_mode_dual_wavelength_results/detector_layout.png
Creating model...
Model created with 1 layers
Total parameters: 16384
Starting training...
Training on device: cuda
Model device: cuda:0
Input fields devices: [device(type='cuda', index=0), device(type='cuda', index=0)]
Epoch [50/100], Loss: -

/home/shiyue/ODNN_MULTIWAVE/SAMPLE/visualizer.py:286: UserWarning: Glyph 25439 (\N{CJK UNIFIED IDEOGRAPH-635F}) missing from font(s) DejaVu Sans.
  plt.savefig(save_path, dpi=300, bbox_inches='tight')
/home/shiyue/ODNN_MULTIWAVE/SAMPLE/visualizer.py:286: UserWarning: Glyph 22833 (\N{CJK UNIFIED IDEOGRAPH-5931}) missing from font(s) DejaVu Sans.
  plt.savefig(save_path, dpi=300, bbox_inches='tight')
/home/shiyue/ODNN_MULTIWAVE/SAMPLE/visualizer.py:286: UserWarning: Glyph 35757 (\N{CJK UNIFIED IDEOGRAPH-8BAD}) missing from font(s) DejaVu Sans.
  plt.savefig(save_path, dpi=300, bbox_inches='tight')
/home/shiyue/ODNN_MULTIWAVE/SAMPLE/visualizer.py:286: UserWarning: Glyph 32451 (\N{CJK UNIFIED IDEOGRAPH-7EC3}) missing from font(s) DejaVu Sans.
  plt.savefig(save_path, dpi=300, bbox_inches='tight')
/home/shiyue/ODNN_MULTIWAVE/SAMPLE/visualizer.py:286: UserWarning: Glyph 26354 (\N{CJK UNIFIED IDEOGRAPH-66F2}) missing from font(s) DejaVu Sans.
  plt.savefig(save_path, dpi=300, bbox_inches='tig

波长分离性能指标图已保存到: ./single_mode_dual_wavelength_results/separation_performance.png

All results saved to: ./single_mode_dual_wavelength_results
Training and visualization completed successfully!
Results summary saved to: ./single_mode_dual_wavelength_results/results_summary.json
